# PECT-JEPA: EXP-70 Huấn Luyện Full 81 Files Cross-Condition Pairing & TTA Evaluation
## Chiến lược 2: Self-Supervised Cross-Condition Pairing trên toàn bộ 81 scans (Stride 4) + Test-Time Adaptation

**Mục tiêu thí nghiệm EXP-70:**
1. **Pretraining Toàn diện (81 files)**:
   - Huấn luyện JEPA trên toàn bộ 81 file TDMS (Corrosion, Rivet_v1, Rivet_v2) gồm 2,106 cặp chéo điều kiện định hướng (A -> B != B -> A).
   - Khắc phục triệt để hạn chế của EXP-63/EXP-69 (vốn chỉ học 16 file Base, bỏ sót hoàn toàn cảm biến TMR và xung Chirp).
   - Tối ưu không gian lấy mẫu: `grid_stride = 4` (nửa bán kính r3=7), giữ tổng số mẫu ở mức ~9.7 triệu patch pairs (973k mẫu/epoch, ~3.5 phút/epoch trên Colab T4).
2. **Bảo toàn cơ chế điều biến an toàn**:
   - Condition token $c_B$ trích xuất từ 4 góc `corner_median` (0% rò rỉ khuyết tật).
   - Dodd-Deeds 16D spectral phase encoding `cos_sin` liên tục trên mặt cầu đơn vị $S^{D-1}$.
   - Unified 64D latent representation + Normalized L1 loss.
3. **Đánh giá Downstream toàn diện (Stride 1)**:
   - Trích xuất toàn bộ 72,900 điểm ảnh mỗi C-scan (stride 1) cho tất cả các bài toán.
   - Anomaly Detection (Linear Probe & Mahalanobis AUC/AP/CNR).
   - True Flaw Sizing Hurdle Protocol ($R^2$, MAE mm trên mẫu Corrosion).
   - Defect Segmentation (IoU, Dice F1).
   - Canonical TTA Benchmark (32 files target trên 4 trục dịch chuyển sensor, waveform, liftoff, specimen).


In [1]:
from google.colab import drive
import os, glob

drive.mount('/content/drive', force_remount=True)

DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)

tdms_files = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print(f'[*] Tìm thấy {len(tdms_files)}/81 files TDMS')

for p in [
    os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
    os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
    os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')
]:
    status = 'OK  ' if os.path.exists(p) else 'MISS'
    print(f'[{status}] {p}')


Mounted at /content/drive
[*] Tìm thấy 81/81 files TDMS
[OK  ] /content/drive/MyDrive/PECT_Dataset/ground_truth/corrosion/corrosion_gt_mask.npy
[OK  ] /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v1/rivet_v1_gt_mask.npy
[OK  ] /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v2/rivet_v2_gt_mask.npy


In [2]:
# Đồng bộ mã nguồn mới nhất từ remote branch Define_PECT_JEPA
!test -d GraduationProject && (cd GraduationProject && git fetch origin && git checkout Define_PECT_JEPA && git reset --hard origin/Define_PECT_JEPA) || git clone -b Define_PECT_JEPA https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git log --oneline -3


Cloning into 'GraduationProject'...
remote: Enumerating objects: 2767, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 2767 (delta 200), reused 168 (delta 107), pack-reused 2498 (from 1)
Receiving objects: 100% (2767/2767), 41.39 MiB | 27.11 MiB/s, done.
Resolving deltas: 100% (1882/1882), done.
3a089a1 (HEAD -> Define_PECT_JEPA, origin/Define_PECT_JEPA) fix(colab): repair JSON syntax error, clean reference pipeline, and add train_70_full_cross_pairing.ipynb
62ae259 feat(data): support full 81-file cross-condition pairing pretraining with grid stride 4
b8a8f20 fix(evaluate): resolve ground truth paths robustly and avoid console encoding crash on Windows


In [3]:
# Cài đặt môi trường chuẩn khớp với phòng thí nghiệm
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm matplotlib joblib pandas 2>&1 | tail -3

import torch, sklearn, scipy, nptdms
print('=' * 60)
print(f'PyTorch:      {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print(f'Scikit-Learn: {sklearn.__version__}')
print('=' * 60)


google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


PyTorch:      2.11.0+cu130 | CUDA: True
GPU Device:   Tesla T4
VRAM:         15.64 GB
Scikit-Learn: 1.7.2


In [4]:
# Symlink thư mục runs nội bộ sang Google Drive để lưu checkpoints và kết quả an toàn
!cd GraduationProject && mkdir -p experiments/cross_condition && if [ ! -L experiments/cross_condition/runs ]; then mv experiments/cross_condition/runs/* "$DRIVE_RUNS/" 2>/dev/null; rm -rf experiments/cross_condition/runs && ln -s "$DRIVE_RUNS" experiments/cross_condition/runs && echo '[OK] LINKED'; else echo '[OK] ALREADY_LINKED'; fi
!ln -sfn "$DRIVE_RUNS" GraduationProject/experiments/cross_condition/runs
!ls -la GraduationProject/experiments/cross_condition/ | grep runs


[OK] LINKED
lrwxrwxrwx 1 root root   41 Oct 23 17:58 runs -> /content/drive/MyDrive/PECT_Dataset/runs/


In [5]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'


# ===== CẤU HÌNH THỰC NGHIỆM EXP-70 (Full 81-File Cross-Condition Pairing + Grid Stride 4) =====
EXP_NAME = 'exp70_full_cross_pairing'

BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement false --unified_latent true --two_stream_backbone false --embed_dim 64 --var_weight 0.5 --cov_weight 0.2 --var_gamma 1.0 --scan_centered_vicreg true --inv_start_epoch 11 --inv_ramp_epochs 20 --inv_cosine true --scan_centered_inv true --inv_weight 0.2 --diverge_patience 0 --device cuda --use_full_dataset true --grid_stride 4'
EXTRA_FLAGS = '--continuous_pos_embed true --contiguous_masking true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --spectral_condition_token true --spectral_phase_encoding cos_sin --use_condition_token true --condition_modulation adaln --pred_loss_type normalized_l1 --anti_hijack_ratio 0.0 --self_focus_gamma 0.0 --ref_patch_mode corner_median'

os.environ.update({
    'EXP_NAME': EXP_NAME,
    'BASE_FLAGS': BASE_FLAGS,
    'EXTRA_FLAGS': EXTRA_FLAGS,
    'DATA': DATA,
    'DRIVE_RUNS': DRIVE_RUNS
})

print(f'[*] Thí nghiệm: {EXP_NAME}')
print(f'[*] Flags: {BASE_FLAGS} {EXTRA_FLAGS}')


[*] Thí nghiệm: exp70_full_cross_pairing
[*] Flags: --star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement false --unified_latent true --two_stream_backbone false --embed_dim 64 --var_weight 0.5 --cov_weight 0.2 --var_gamma 1.0 --scan_centered_vicreg true --inv_start_epoch 11 --inv_ramp_epochs 20 --inv_cosine true --scan_centered_inv true --inv_weight 0.2 --diverge_patience 0 --device cuda --use_full_dataset true --grid_stride 4 --continuous_pos_embed true --contiguous_masking true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --spectral_condition_token true --spectral_phase_encoding cos_sin --use_condition_token true --condition_modulation adaln --pred_loss_type normalized_l1 --anti_hijack_ratio 0.0 --self_focus_gamma 0.0 --ref_patch_mode corner_median


In [6]:
# Huấn luyện mô hình PECT-JEPA từ đầu (100 epochs, eval val mỗi epoch)
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train \
    --exp_name "$EXP_NAME" \
    --add_timestamp true \
    $BASE_FLAGS \
    $EXTRA_FLAGS \
    --data_dir "$DATA" \
    --epochs 200 \
    --batch_size 256 \
    --eval_val_every 1


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
[17:58:54 | INFO] Initialized Unified Experiment Directory: experiments/cross_condition/runs/exp70_full_cross_pairing_20261023_175853
[17:58:54 | INFO] Configuration: CrossConditionConfig(exp_name='exp70_full_cross_pairing', add_timestamp=True, base_exp_dir='experiments/cross_condition/runs', embed_dim=64, encoder_depth=4, encoder_heads=4, predictor_depth=2, predictor_heads=4, mlp_ratio=4.0, dropout=0.0, spatial_topology='concentric_star', star_radii=(2, 4, 7), grid_size=5, temporal_samples=128, num_harmonics=16, num_target_probes=9, num_context_probes=16, num_target_rays=3, continuous_pos_embed=True, contiguous_masking=True, predictor_scattering_only=False, scan_centered_inv=True, patch_energy_norm=False, median_patch_energy=False, clean_phasor_token=False, subspace_norm=False, 

In [ ]:
# [TÙY CHỌN] Chạy cell này CHỈ KHI muốn tiếp tục quá trình train bị gián đoạn từ Google Drive
import os, glob

candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")), key=os.path.getmtime)
if not candidates:
    raise FileNotFoundError(f'Không tìm thấy thư mục run nào cho {EXP_NAME} trong {DRIVE_RUNS}')
run_dir = candidates[-1]

ckpt_latest = os.path.join(run_dir, 'checkpoints', 'latest_cross_condition_jepa.pt')
ckpt_best = os.path.join(run_dir, 'checkpoints', 'best_cross_condition_jepa.pt')
resume_ckpt = ckpt_latest if os.path.exists(ckpt_latest) else ckpt_best

print(f'[*] Resume từ checkpoint: {resume_ckpt}')
os.environ['RESUME_CKPT'] = resume_ckpt

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train \
    --exp_name "$EXP_NAME" \
    $BASE_FLAGS \
    $EXTRA_FLAGS \
    --data_dir "$DATA" \
    --epochs 200 \
    --batch_size 256 \
    --eval_val_every 1 \
    --resume_checkpoint "$RESUME_CKPT"


In [7]:
# ==============================================================================
# Huấn luyện probe downstream (Linear/MLP Detection & Ridge/MLP Sizing)
# Bộ mô hình bao gồm: Raw Probes (Baseline) và TTA Whitened Probes (Test-Time Adaptation)
# ==============================================================================
import os, glob

candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")), key=os.path.getmtime)
if not candidates:
    raise FileNotFoundError(f'Không tìm thấy run folder cho {EXP_NAME} trong {DRIVE_RUNS}')
run_dir = candidates[-1]

ckpt_best = os.path.join(run_dir, 'checkpoints', 'best_cross_condition_jepa.pt')
ckpt_latest = os.path.join(run_dir, 'checkpoints', 'latest_cross_condition_jepa.pt')
eval_ckpt = ckpt_best if os.path.exists(ckpt_best) else ckpt_latest

downstream_dir = os.path.join(run_dir, 'downstream_models')
os.makedirs(downstream_dir, exist_ok=True)

os.environ['EVAL_CKPT'] = eval_ckpt
os.environ['DOWNSTREAM_DIR'] = downstream_dir
os.environ['RUN_DIR'] = run_dir

print(f'[*] Checkpoint đánh giá: {eval_ckpt}')
print(f'[*] Thư mục lưu probe:  {downstream_dir}')

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.downstream.train_downstream \
    --checkpoint "$EVAL_CKPT" \
    --data_dir "$DATA" \
    --out_dir "$DOWNSTREAM_DIR" \
    --device cuda \
    --tta_trim_ratio 0.75 \
    --tta_shrinkage 0.01


[*] Checkpoint đánh giá: /content/drive/MyDrive/PECT_Dataset/runs/exp70_full_cross_pairing_20261023_175853/checkpoints/best_cross_condition_jepa.pt
[*] Thư mục lưu probe:  /content/drive/MyDrive/PECT_Dataset/runs/exp70_full_cross_pairing_20261023_175853/downstream_models
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
CANONICAL DOWNSTREAM & TEST-TIME ADAPTATION (TTA) TRAINING PIPELINE
  Checkpoint:       /content/drive/MyDrive/PECT_Dataset/runs/exp70_full_cross_pairing_20261023_175853/checkpoints/best_cross_condition_jepa.pt
  Output Dir:       /content/drive/MyDrive/PECT_Dataset/runs/exp70_full_cross_pairing_20261023_175853/downstream_models
  TTA Trim Ratio:   0.75
  TTA Shrinkage:    0.01
  Device:           cuda

[STEP 1/2] Extracting Source Domain Pool (16 Base Domain files: Train + Val)...

  Source Detection samples: 60

In [8]:
# ==============================================================================
# Đánh giá toàn diện 4 tasks trên toàn bộ 81 scans (12 Train, 4 Val, 65 Test OOD)
# 1. Task 1: Anomaly Detection (xuất ảnh 4 subplots: CAD GT, Raw Vpp, Mahalanobis, Linear Probe)
# 2. Task 2: Flaw Sizing Hurdle Protocol (R^2, MAE mm, RMSE mm kèm scatter plot)
# 3. Task 3: Defect Segmentation (IoU, Dice F1, Precision, Recall)
# 4. Task 4: Canonical Test-Time Adaptation (TTA) Benchmark (xuất ảnh 5 subplots: CAD GT, Raw Vpp, Zero-Shot LP, TTA LP, TTA Mahalanobis)
# ==============================================================================
eval_out_dir = os.path.join(run_dir, 'evaluation_results')
os.environ['EVAL_OUT_DIR'] = eval_out_dir

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluate \
    --checkpoint "$EVAL_CKPT" \
    --data_dir "$DATA" \
    --downstream_dir "$DOWNSTREAM_DIR" \
    --out_dir "$EVAL_OUT_DIR" \
    --device cuda \
    --eval_tta true \
    --tta_trim_ratio 0.75 \
    --tta_shrinkage 0.01


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (

PECT-JEPA Unified Downstream Benchmark Evaluation
Checkpoint:       /content/drive/MyDrive/PECT_Dataset/runs/exp70_full_cross_pairing_20261023_175853/checkpoints/best_cross_condition_jepa.pt
Output Directory: /content/drive/MyDrive/PECT_Dataset/runs/exp70_full_cross_pairing_20261023_175853/evaluation_results
Compute Device:   cuda
Target Splits:    ['train', 'val', 'test']

Discovered 81 total inspection files:
  - Train Split (Base Domain):     12 files
  - Val Split (Base Domain):       4 files
  - Test Split (Held-out OOD):     65 files

STEP 1: Downstream Probes & TTA Models Setup
  [INFO] Loading pre-trained downstream models from: /content/drive/MyDrive/PECT_Dataset/runs/exp70_full_cross_pairing_20261023_175853/downstream_models
  Pre-trained standard downstream models loa

In [9]:
# ==============================================================================
# Chẩn đoán sâu hình học không gian tiềm ẩn và các cơ chế hiệu chuẩn per-scan (P0, P1, P1u_s2, Otsu)
# ==============================================================================
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.diagnose_readouts \
    --checkpoint "$EVAL_CKPT" \
    --data_dir "$DATA" \
    --device cuda


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (

Pool: 46900 samples (9060 defect)

Halo-cleaned P0 pool: 46900 (9060 pos; removed 0 halo/ring samples)
P2 pool: 45300 (9060 pos, 24000 hardneg, 12240 randneg)
P2h pool: 45300 (9060 pos, 24000 hardneg-clean)
MLP probes fitted: raw(official) + P1-whitened + P1u_s2-whitened




==== READOUT DIAGNOSTICS v6 ====
[TRAIN] n=12 P0=0.9658/0.5696 P1=0.9473/0.4945 P2=0.9172/0.4621 P3=0.8832/0.4721 maha=0.8633 iou05=0.1468/0.1097 iouOpt=0.3732/0.3518
[TRAIN-v3/v6] n=12 P1_s2=0.9612/0.5159 P1u=0.9580/0.4962 P1u_s2=0.9643/0.5356 P1u_trim_s2=0.9631/0.5302 P1u_s5=0.9466/0.5100 P1u_maha=0.9284/0.4083 P0h=0.9660/0.5626 P2h=0.9183/0.4653 iouOtsu=0.1173/f1Otsu=0.2052
[TRAIN-v7_test] n=12 P1u_abs=0.9342/0.4640 P1u_skew=0.9580/0.4962 P1u_s2_abs=0.9413/0.4959 P1u_s2_skew=0.9643/0.5356 P1u_s2_maha=0.93

In [10]:
# ==============================================================================
# BẢNG TỔNG HỢP KẾT QUẢ ĐỊNH LƯỢNG (DOWNSTREAM MASTER BENCHMARK & TTA)
# ==============================================================================
import json
import pandas as pd
from IPython.display import display, HTML

summary_path = os.path.join(eval_out_dir, 'master_benchmark_summary.json')
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    
    print('=' * 80)
    print(' MASTER BENCHMARK EVALUATION SUMMARY')
    print('=' * 80)
    
    # 1. Bảng theo Splits
    splits_data = []
    for sp, data in summary.get('by_split', {}).items():
        splits_data.append({
            'Split': sp.upper(),
            'Files': data.get('count', 0),
            'LP AUC': f"{data.get('linear_probe_auc_mean', 0.0):.4f} ± {data.get('linear_probe_auc_std', 0.0):.4f}",
            'LP AP': f"{data.get('linear_probe_ap_mean', 0.0):.4f}",
            'Maha AUC': f"{data.get('mahalanobis_auc_mean', 0.0):.4f}",
            'CNR': f"{data.get('cnr_mean', 0.0):.2f}",
            'Seg IoU': f"{data.get('segmentation_iou_mean', 0.0):.4f}",
            'Seg Dice': f"{data.get('segmentation_dice_mean', 0.0):.4f}",
            'Sizing R²': f"{data.get('sizing_r2_mean', 0.0):.4f}",
            'MAE (mm)': f"{data.get('sizing_mae_mean', 0.0):.3f}"
        })
    df_splits = pd.DataFrame(splits_data)
    display(HTML('<h3>1. Hiệu năng theo Tập dữ liệu (Splits)</h3>'))
    display(df_splits)
    
    # 2. Bảng theo Cảm biến (Sensor Shift)
    sensors_data = []
    for sn, data in summary.get('by_sensor', {}).items():
        sensors_data.append({
            'Sensor': sn,
            'Files': data.get('count', 0),
            'LP AUC': f"{data.get('linear_probe_auc_mean', 0.0):.4f} ± {data.get('linear_probe_auc_std', 0.0):.4f}",
            'LP AP': f"{data.get('linear_probe_ap_mean', 0.0):.4f}",
            'Maha AUC': f"{data.get('mahalanobis_auc_mean', 0.0):.4f}",
            'CNR': f"{data.get('cnr_mean', 0.0):.2f}",
            'Seg IoU': f"{data.get('segmentation_iou_mean', 0.0):.4f}"
        })
    df_sensors = pd.DataFrame(sensors_data)
    display(HTML('<h3>2. Hiệu năng theo Loại Cảm biến (Sensor)</h3>'))
    display(df_sensors)

    # 3. Bảng Canonical Test-Time Adaptation (TTA) Benchmark
    tta_res = summary.get('canonical_test_time_adaptation', {})
    if tta_res and 'shifts' in tta_res:
        tta_data = []
        for shift_axis, data in tta_res['shifts'].items():
            base = data.get('baseline_zero_shot', {})
            tta_lp = data.get('test_time_adapted_lp', {})
            tta_mh = data.get('test_time_adapted_maha', {})
            delta_lp = data.get('deltas', {}).get('delta_lp_auc', 0.0)
            delta_mh = data.get('deltas', {}).get('delta_maha_auc', 0.0)
            tta_data.append({
                'Shift Axis': shift_axis.upper(),
                'Target Files': data.get('target_count', 0),
                'Zero-Shot LP AUC': f"{base.get('mean_auc', 0.0):.4f}",
                'TTA LP AUC': f"{tta_lp.get('mean_auc', 0.0):.4f}",
                'Δ LP AUC': f"{delta_lp:+.4f}",
                'TTA Maha AUC': f"{tta_mh.get('mean_auc', 0.0):.4f}",
                'Δ Maha AUC': f"{delta_mh:+.4f}",
                'TTA LP AP': f"{tta_lp.get('mean_ap', 0.0):.4f}"
            })
        df_tta = pd.DataFrame(tta_data)
        display(HTML('<h3>3. Kết quả Canonical Test-Time Adaptation (TTA) Benchmark</h3>'))
        display(df_tta)
else:
    print(f'[!] Chưa tìm thấy {summary_path}')


 MASTER BENCHMARK EVALUATION SUMMARY


,Split,Files,LP AUC,LP AP,Maha AUC,CNR,Seg IoU,Seg Dice,Sizing R²,MAE (mm)
0,TRAIN,12,0.9653 ± 0.0413,0.5574,0.8633,1.53,0.1356,0.2329,0.4438,1.369
1,VAL,4,0.9866 ± 0.0118,0.4803,0.9085,1.82,0.1271,0.2249,0.4805,1.319
2,TEST,65,0.8290 ± 0.1819,0.3525,0.8180,1.35,0.1087,0.1884,0.1327,1.724


,Sensor,Files,LP AUC,LP AP,Maha AUC,CNR,Seg IoU
0,Hall_Air_Core,27,0.8414 ± 0.1898,0.4132,0.8130,1.32,0.1043
1,Hall_Pot_Core,27,0.8861 ± 0.1381,0.4422,0.8841,1.60,0.1455
2,TMR,27,0.8436 ± 0.1833,0.3121,0.7905,1.27,0.0911


In [11]:
# ==============================================================================
# HIỂN THỊ TRỰC QUAN C-SCAN ANOMALY DETECTION & TTA MAPS
# ==============================================================================
import glob
from PIL import Image
import matplotlib.pyplot as plt

# 1. Hiển thị Anomaly Detection (4-panel subplots: CAD GT, Raw Vpp, Mahalanobis, Linear Probe)
anom_plots = sorted(glob.glob(os.path.join(eval_out_dir, 'task_1_anomaly_detection', '**', '*.png'), recursive=True))
print(f'[*] Tìm thấy {len(anom_plots)} biểu đồ Anomaly Detection.')

if anom_plots:
    for p in anom_plots[:3]:
        img = Image.open(p)
        plt.figure(figsize=(18, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'Anomaly Detection 4-Panel Map: {os.path.basename(p)}', fontsize=12, fontweight='bold')
        plt.show()

# 2. Hiển thị TTA Comparison (5-panel subplots: CAD GT, Raw Vpp, Zero-Shot LP, TTA LP, TTA Maha)
tta_plots = sorted(glob.glob(os.path.join(eval_out_dir, 'task_4_canonical_tta', '*.png')))
print(f'[*] Tìm thấy {len(tta_plots)} biểu đồ TTA Benchmark.')

if tta_plots:
    for p in tta_plots[:3]:
        img = Image.open(p)
        plt.figure(figsize=(22, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'TTA Adaptation 5-Panel Map: {os.path.basename(p)}', fontsize=12, fontweight='bold')
        plt.show()

# 3. Hiển thị Sizing Calibration Scatter Plot
sizing_plots = sorted(glob.glob(os.path.join(eval_out_dir, 'task_2_sizing_hurdle', '**', '*.png'), recursive=True))
if sizing_plots:
    img = Image.open(sizing_plots[0])
    plt.figure(figsize=(8, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Sizing Hurdle Calibration Scatter: {os.path.basename(sizing_plots[0])}', fontsize=12, fontweight='bold')
    plt.show()


[*] Tìm thấy 0 biểu đồ Anomaly Detection.
[*] Tìm thấy 0 biểu đồ TTA Benchmark.


In [12]:
# ==============================================================================
# KIỂM TRA ĐỒNG BỘ TOÀN VẸN TRÊN GOOGLE DRIVE
# ==============================================================================
print('=' * 80)
print(f'Kiểm tra thư mục kết quả: {run_dir}')
for root, dirs, files in os.walk(run_dir):
    level = root.replace(run_dir, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')
    if level <= 2:
        for f in files[:5]:
            fpath = os.path.join(root, f)
            sz = os.path.getsize(fpath) / (1024 * 1024)
            print(f'{indent}    - {f} ({sz:.2f} MB)')
        if len(files) > 5:
            print(f'{indent}    ... và {len(files) - 5} files khác')
print('=' * 80)
print('[OK] Toàn bộ dữ liệu huấn luyện, weights và kết quả đánh giá đã được đồng bộ an toàn trên Google Drive!')


Kiểm tra thư mục kết quả: /content/drive/MyDrive/PECT_Dataset/runs/exp70_full_cross_pairing_20261023_175853
exp70_full_cross_pairing_20261023_175853/ (4 files)
    - run.log (0.03 MB)
    - config.json (0.00 MB)
    - git_info.json (0.00 MB)
    - metrics_epoch.csv (0.02 MB)
    checkpoints/ (2 files)
        - best_cross_condition_jepa.pt (5.41 MB)
        - latest_cross_condition_jepa.pt (5.42 MB)
    results/ (0 files)
    visualizations/ (1 files)
        - training_curves.png (0.21 MB)
    downstream_models/ (2 files)
        - standard_probes.joblib (0.43 MB)
        - downstream_train_summary.json (0.00 MB)
    evaluation_results/ (3 files)
        - master_benchmark_metrics.csv (0.02 MB)
        - master_benchmark_summary.json (0.03 MB)
        - EVALUATION_REPORT.md (0.00 MB)
        1_Anomaly_Detection/ (0 files)
            train/ (38 files)
            val/ (14 files)
            test/ (197 files)
        2_Defect_Sizing/ (0 files)
            train/ (9 files)
            v